# RG3 시계열 모델 및 RF 앙상블
원본32샷 이력, 겹친학습윈도우제외, 테스트정답사용금지.

In [1]:
from pathlib import Path
import json,warnings
import numpy as np,pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,confusion_matrix
from sktime.transformations.rocket import Rocket,MiniRocketMultivariate,MultiRocketMultivariate
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
dataset=d.copy(); label_y=y.copy(); label_groups=groups.copy(); label_blocks=blocks.copy()
prep=''.join(json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))['cells'][1]['source']).split('    for feature in features:')[0]
prep=prep.split('for feature in features:')[0]
exec(prep)
d=dataset;y=label_y;groups=label_groups;blocks=label_blocks
channels=['Barrel_Temperature_2','Barrel_Temperature_5','Hopper_Temperature','Cushion_Position','Max_Injection_Pressure','Average_Back_Pressure','Max_Injection_Speed','Filling_Time']
lookup={(r['EQUIP_CD'],r['product'],r['TimeStamp']):i for i,r in observed[keys].iterrows()}
segment_ids={s:g.index.to_numpy() for s,g in observed.groupby('history_segment')}
windows=[];record_sets=[];coverage=[]
for r in d.itertuples():
    idx=lookup[(r.EQUIP_CD,r.product,r.TimeStamp)]
    ids=segment_ids[observed.history_segment.iloc[idx]]
    pos=np.searchsorted(ids,idx);selected=ids[max(0,pos-31):pos+1]
    assert observed.TimeStamp.iloc[selected].max()<=r.TimeStamp
    arr=observed.iloc[selected][channels].to_numpy(float)
    assert np.allclose(arr[-1],np.array([getattr(r,c) for c in channels]),equal_nan=True)
    windows.append(np.pad(arr,((32-len(arr),0),(0,0)),mode='edge').T)
    record_sets.append(set(selected));coverage.append(dict(TimeStamp=r.TimeStamp,actual=int(r.PassOrFail),available_shots=len(arr)))
X=np.stack(windows)
pd.DataFrame(coverage).to_csv(tables/'rg3_sequence32_coverage.csv',index=False,encoding='utf-8-sig')
factories={'ROCKET':lambda:Rocket(num_kernels=1000,normalise=False,n_jobs=4,random_state=42),'MiniRocket':lambda:MiniRocketMultivariate(num_kernels=1008,n_jobs=4,random_state=42),'MultiRocket':lambda:MultiRocketMultivariate(num_kernels=168,n_features_per_kernel=np.int32(4),n_jobs=4,random_state=42)}
cache_ts={};audit=[];warning_rows=[]
def time_predict(name,ref,q):
    key=(name,tuple(ref),tuple(q))
    if key in cache_ts:return cache_ts[key]
    query_records=set().union(*(record_sets[z] for z in q))
    safe=np.array([z for z in ref if record_sets[z].isdisjoint(query_records)],dtype=int)
    audit.append(dict(model=name,ref_before=len(ref),ref_after_purge=len(safe),ref_gas=int(y[safe].sum()),query=len(q)))
    if len(np.unique(y[safe]))<2:
        score=np.full(len(q),float(y[safe][0]) if len(safe) else 0.)
    else:
        tr=X[safe].copy();te=X[q].copy()
        med=np.nanmedian(tr,axis=(0,2))
        for c in range(len(channels)):
            tr[:,c,:]=np.nan_to_num(tr[:,c,:],nan=med[c])
            te[:,c,:]=np.nan_to_num(te[:,c,:],nan=med[c])
        # Shape rather than absolute product level; per-window initial16 median.
        tr=tr-np.median(tr[:,:,:16],axis=2,keepdims=True)
        te=te-np.median(te[:,:,:16],axis=2,keepdims=True)
        scale=np.maximum(np.std(tr,axis=(0,2)),1e-6)
        tr=(tr/scale[None,:,None]).astype(np.float32);te=(te/scale[None,:,None]).astype(np.float32)
        if name=='MultiRocket':tr=tr.astype(np.float64);te=te.astype(np.float64)
        transform=factories[name]()
        with warnings.catch_warnings(record=True) as ww:
            warnings.simplefilter('always')
            a=transform.fit_transform(tr);b=transform.transform(te)
            learner=make_pipeline(StandardScaler(),LogisticRegression(C=.1,class_weight='balanced',max_iter=2000,random_state=42))
            score=learner.fit(a,y[safe]).predict_proba(b)[:,1]
            warning_rows.extend([dict(model=name,message=str(w.message)) for w in ww])
    assert np.isfinite(score).all()
    cache_ts[key]=score
    return score
rows=[];failed=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=[];inner=[]
        for name in factories:
            outer.append(time_predict(name,ref,q))
            cal=np.full(len(ref),np.nan)
            for ib in groups.iloc[ref].unique():
                mask=groups.iloc[ref].eq(ib).to_numpy()
                if (~mask).any():cal[mask]=time_predict(name,ref[~mask],ref[mask])
            inner.append(cal)
        outer=np.column_stack(outer);inner=np.column_stack(inner)
        rf_outer=pair_predict(ref,q).max(axis=1);rf_inner=np.full(len(ref),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():rf_inner[mask]=pair_predict(ref[~mask],ref[mask]).max(axis=1)
        candidates={name:(outer[:,k],inner[:,k]) for k,name in enumerate(factories)}
        candidates['TS3_mean']=(outer.mean(axis=1),inner.mean(axis=1))
        candidates['RF_TS3_mean']=((rf_outer+outer.mean(axis=1))/2,(rf_inner+inner.mean(axis=1))/2)
        candidates['RF_TS3_max']=(np.maximum(rf_outer,outer.mean(axis=1)),np.maximum(rf_inner,inner.mean(axis=1)))
        for name,(score,cal) in candidates.items():
            normal=cal[(y[ref]==0)&np.isfinite(cal)]
            for budget in [.01,.03,.05]:
                threshold=float(np.quantile(normal,1-budget,method='higher')) if len(normal) else 1.
                for i,z in enumerate(q):rows.append(dict(scheme=scheme,model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[i]),threshold=threshold,prediction=int(score[i]>threshold)))
        print('Completed sequence',scheme,block,flush=True)
pp=pd.DataFrame(rows);result=[]
for (scheme,name,budget),g in pp.groupby(['scheme','model','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    result.append(dict(scheme=scheme,model=name,budget=budget,shots=len(g),gas=int(g.actual.sum()),AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn)))
metrics=pd.DataFrame(result)
metrics.to_csv(tables/'rg3_sequence_ensembles_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_sequence_ensembles_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_sequence_ensembles_purge_audit.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(warning_rows,columns=['model','message']).to_csv(tables/'rg3_sequence_ensembles_warnings.csv',index=False,encoding='utf-8-sig')
summary=metrics.loc[metrics.budget.eq(.01)].sort_values(['scheme','AP'],ascending=[True,False])
lines=['# RG3 가스 시계열 분류와 RF 앙상블','',
'샷 내부 압력파형이 아니라 샷 간 센서값의 시계열이다. 현재샷 포함32샷×8센서, 미래값 미사용. 30분 단절/제품전환으로 원본이력 구간을 끊는다. 짧은 이력은 가장 이른 관측값을 왼쪽에 복제하고583개/가스17 유지한다.',
'각 학습 시 평가윈도우와 원본 샷이 겹치는 학습윈도우는 제거(purge)한다. 결측대체·센서scale·ROCKET 변환은 학습집합에서만 적합한다. 각 윈도우 초기16샷중앙값을 빼서 절대센서수준보다 변화모양을 본다. padding/중심화는실시간 현재샷까지 관측가능 데이터만 사용한다.',
'ROCKET1000,MiniRocket1008,MultiRocket168커널,분류기는C0.1balancedLogistic 고정. 최신설치 구현이지만 모든 알고리즘이최신발표는아니다. 커널수가 작으므로 전체 알고리즘 최적성능을 대표하지 않는다.',
'TS3_mean은세시계열모델동일평균,RF_TS3_mean은기존RF두신호max와TS3_mean의평균,RF_TS3_max는두점수최대. 결합및기준은평가정답으로조정하지않는다. 점수는독립확률보정하지않음.',
'기준은학습내부구간검증정상상위1/3/5%경계이며평가오탐률보장아님. RF계열이력 purge는기존2신호정의유지로시계열채널purge와동일하지않아별도비교조건이다. 원본이력테스트제공미확인/동일자료반복탐색/최종독립평가아님.',
'','| 평가 | 모델/결합 | AP | TP | FP | FN | TN |','|---|---|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} |')
(root/'reports/rg3_sequence_ensemble_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(summary.to_string(index=False))


             scheme       model  budget  shots  gas       AP  TP  FP  FN  TN
leave_one_block_out  RF_TS3_max    0.01    583   17 0.165297   4   6  13 560
leave_one_block_out RF_TS3_mean    0.01    583   17 0.099150   3  15  14 551
leave_one_block_out      ROCKET    0.01    583   17 0.043460   1  23  16 543
leave_one_block_out    TS3_mean    0.01    583   17 0.037331   1  38  16 528
leave_one_block_out  MiniRocket    0.01    583   17 0.036322   1  35  16 531
leave_one_block_out MultiRocket    0.01    583   17 0.030773   1  18  16 548
   past_blocks_only  RF_TS3_max    0.01    440   13 0.078618   2   4  11 423
   past_blocks_only RF_TS3_mean    0.01    440   13 0.067239   1   6  12 421
   past_blocks_only MultiRocket    0.01    440   13 0.039243   1  16  12 411
   past_blocks_only  MiniRocket    0.01    440   13 0.038637   2  17  11 410
   past_blocks_only    TS3_mean    0.01    440   13 0.037435   0  10  13 417
   past_blocks_only      ROCKET    0.01    440   13 0.034590   0   1  13 426